# Week 13 · Notebook 2 — Serving an Agent with Flask
**CSCI 250 · Fall 2026**

Put the agent loop behind an HTTP endpoint so others can use it. The same code is in `code/agent_app.py` (run it as a real script). 

> Colab can't easily hold a long-running server, so here we **test the Flask app in-process** with its test client — no network, no key needed.

## 1. The app factory
`run_agent` is pluggable: use the real Claude loop if a key is present, else the fallback. The route just passes the message through.

In [ ]:
import os
from flask import Flask, request, jsonify

def make_agent():
    """Return a run_agent(message)->str, real or fallback."""
    if os.environ.get('ANTHROPIC_API_KEY'):
        # from Notebook 1 you'd import run_claude_agent; inline stub here:
        def run_agent(message):
            return 'TODO: wire to run_claude_agent(message)'
    else:
        def run_agent(message):
            total = 0
            import re
            m = re.search(r'(\d+)\s*\+\s*(\d+)', message)
            if m: total = int(m.group(1)) + int(m.group(2))
            return f'(fallback) sum={total}'
    return run_agent

def create_app():
    app = Flask(__name__)
    run_agent = make_agent()
    @app.route('/ask', methods=['POST'])
    def ask():
        msg = request.get_json(force=True).get('message', '')
        return jsonify({'answer': run_agent(msg)})
    @app.route('/health')
    def health():
        return jsonify({'ok': True})
    return app

## 2. Test it in-process (no server, no key)
Flask's test client lets us POST to `/ask` without starting a real server.

In [ ]:
app = create_app()
client = app.test_client()
print('health:', client.get('/health').get_json())
r = client.post('/ask', json={'message': 'What is 12 + 30?'})
print('ask:', r.get_json())

## 3. Run it for real (outside Colab)
Locally, `python agent_app.py` then in another terminal:
```bash
curl -X POST localhost:5000/ask -H 'Content-Type: application/json' \
     -d '{"message": "What is 12 + 30, and the price of coffee?"}'
```
**Never hard-code your API key in the app** — it reads it from the environment. For A10, wire `run_agent` to your real tool-using loop from Notebook 1.

In [ ]:
# scratch:
